# 🧪 08 · AB 实验与显著性检验

> 业务里「新功能有没有用」靠 AB 实验说话。本节从业务视角讲**假设 → 检验 → 样本量 → AA/SRM → peeking 陷阱**，手写 z/t 检验并与 scipy 对照。
> （更严格的统计推导与因果推断见 [17-数据科学](../17-数据科学/README.md) 模块；本节侧重业务落地）

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 转化率差异显著吗？ | z 检验（比率） |
| 2 | 平均值差异显著吗？ | t 检验 |
| 3 | 样本量要多大？ | 功效公式 |
| 4 | 分组是否均匀？ | AA 检验 / SRM |
| 5 | 看结果看太多次会怎样？ | peeking 陷阱 |

In [1]:
import numpy as np
import scipy.stats as st
rng = np.random.default_rng(17)

# 模拟 AB：对照组转化率 p=0.10，实验组 p=0.12，每组 3000 人
n = 3000
ctrl = rng.binomial(1, 0.10, n)
exp  = rng.binomial(1, 0.12, n)
print(f'对照组转化率: {ctrl.mean():.4f}  实验组转化率: {exp.mean():.4f}')
print(f'绝对提升: {(exp.mean()-ctrl.mean()):.4f}  相对提升: {(exp.mean()/ctrl.mean()-1):.2%}')

对照组转化率: 0.0990  实验组转化率: 0.1160
绝对提升: 0.0170  相对提升: 17.17%


## 1️⃣ 转化率差异：z 检验（两比例检验）

In [2]:
# 两比例 z 检验：H0: p1 == p2
p1, p2 = ctrl.mean(), exp.mean()
p_pool = (ctrl.sum() + exp.sum()) / (2 * n)
se = np.sqrt(p_pool * (1 - p_pool) * (2 / n))
z = (p2 - p1) / se
p_value = 2 * (1 - st.norm.cdf(abs(z)))

# scipy 对照
z_chi2, p_chi2, _, _ = st.chi2_contingency([[ctrl.sum(), n-ctrl.sum()],
                                            [exp.sum(), n-exp.sum()]])
print(f'手写: z = {z:.3f}, p 值 = {p_value:.4f}')
print(f'scipy 卡方(等价): p 值 = {p_chi2:.4f}')
assert p_value < 0.05
print(f'\n✅ p < 0.05 → 差异显著，可以拒绝 H0（实验组转化率显著更高）')

手写: z = 2.126, p 值 = 0.0335
scipy 卡方(等价): p 值 = 0.0372

✅ p < 0.05 → 差异显著，可以拒绝 H0（实验组转化率显著更高）


## 2️⃣ 均值差异：t 检验（如人均时长、客单价）

In [3]:
# 实验组人均时长提升 5 分钟，组内波动 σ=18
dur_ctrl = rng.normal(180, 18, n)
dur_exp  = rng.normal(185, 18, n)

t_stat, p_t = st.ttest_ind(dur_exp, dur_ctrl)
print(f't = {t_stat:.3f}, p 值 = {p_t:.4f}')
assert p_t < 0.05

# 95% 置信区间（均值差）
se_diff = np.sqrt(18**2/n + 18**2/n)
diff_mean = dur_exp.mean() - dur_ctrl.mean()
ci = (diff_mean - 1.96*se_diff, diff_mean + 1.96*se_diff)
print(f'均值差 = {diff_mean:.2f} 分钟，95% CI = [{ci[0]:.2f}, {ci[1]:.2f}]')
print('✅ CI 不含 0，结论与 t 检验一致')

t = 11.310, p 值 = 0.0000
均值差 = 5.26 分钟，95% CI = [4.35, 6.17]
✅ CI 不含 0，结论与 t 检验一致


## 3️⃣ 样本量估算（功效分析）

In [4]:
# 两比例样本量：n ≈ (z_α/2 + z_β)² · (p1(1-p1)+p2(1-p2)) / δ²
alpha, power = 0.05, 0.80
z_alpha = st.norm.ppf(1 - alpha/2)
z_beta  = st.norm.ppf(power)
p1, p2 = 0.10, 0.12
delta = p2 - p1
n_needed = (z_alpha + z_beta)**2 * (p1*(1-p1) + p2*(1-p2)) / delta**2
print(f'需要样本量(每组): {np.ceil(n_needed):.0f}')

# 说明：n 随 α↓、β↓、效应 δ↓、方差↑ 而增大
print('\n➡️ 关系：δ 越小要越多样本；要求功效越高要越多样本；波动越大要越多样本')

需要样本量(每组): 3839

➡️ 关系：δ 越小要越多样本；要求功效越高要越多样本；波动越大要越多样本


## 4️⃣ AA 检验：分组是否真的均匀

In [5]:
# AA 检验：两组的旧指标应无显著差异，否则说明分流有问题（SRM）
aa1 = rng.normal(100, 15, 1000)
aa2 = rng.normal(100, 15, 1000)
t_aa, p_aa = st.ttest_ind(aa1, aa2)
print(f'AA 检验: t={t_aa:.3f}, p={p_aa:.3f} → {"通过(无显著差异)" if p_aa>0.05 else "异常!"}')

# SRM：实际分流比例若偏离 50/50 太多，说明用户被系统性地分到某组
n_total = 20000
actual_a = rng.binomial(n_total, 0.5)
chi2_srm, p_srm = st.chisquare([actual_a, n_total-actual_a], f_exp=[n_total/2, n_total/2])
print(f'SRM 卡方: p={p_srm:.3f} → 分流比例正常' if p_srm>0.01 else 'SRM 异常')
assert p_aa > 0.05
print('✅ AA 通过')

AA 检验: t=-0.409, p=0.683 → 通过(无显著差异)
SRM 卡方: p=0.641 → 分流比例正常
✅ AA 通过


## 5️⃣ peeking 陷阱：看结果看太多次

In [6]:
# 演示：如果每次新增 200 用户就看一次显著性，假阳性率会飙升
# 模拟 H0 为真（两组无差异），但反复 peek
n_peeks = 10
trial = 500
false_pos = 0
for _ in range(trial):
    hit = False
    for k in range(1, n_peeks+1):
        a = rng.normal(0, 1, 200*k)
        b = rng.normal(0, 1, 200*k)
        _, p = st.ttest_ind(a, b)
        if p < 0.05:
            hit = True
    false_pos += hit
print(f'peek 10 次，假阳性率: {false_pos/trial:.1%}（固定 α=5%，应约为 5%）')
assert false_pos/trial > 0.15
print('\n✅ 演示成功：连续监控会显著抬高假阳性率 → 事前定样本量与观察期')

peek 10 次，假阳性率: 41.2%（固定 α=5%，应约为 5%）

✅ 演示成功：连续监控会显著抬高假阳性率 → 事前定样本量与观察期


## 6️⃣ 完整 AB 实验流程（必背）

1. **定假设**：H0 无差异 / H1 有提升；选好主指标与护栏指标；
2. **定样本量**：按 α、β、MDE、基线波动估算，预留观察天数；
3. **随机分流**：用户级/设备级，互斥且均匀（AA/SRM 校验）；
4. **跑实验**：固定时长，不提前看结果（防 peeking）；
5. **分析**：显著性 + 效应量 + 置信区间 + 分层看（防辛普森悖论）；
6. **决策**：显著且业务可接受 → 上线；不显著 → 复盘原因。

> 面试高频追问：样本量公式怎么推、SRM 怎么查、peeking 为什么不行、delta 怎么定、护栏指标是什么。

## 📝 自测清单

- [ ] 能写出两比例 z 检验统计量并解释 p 值
- [ ] 能说出 t 检验与 z 检验的适用场景
- [ ] 能按 α/β/δ/σ 说出样本量的依赖关系
- [ ] 能解释 AA/SRM 的作用与异常后果
- [ ] 能解释 peeking 为什么会抬高假阳性
- [ ] 能背出 AB 实验六步流程